# Query Calibration & Taxonomy Audit

**Author:** Samuele Tommasi

## 1. Notebook Objective

This notebook calibrates the patent retrieval strategy for the Unmanned Aerial Vehicle (UAV) sector. All queries are restricted to substantive Patents of Invetion (excluding utility models) filed between 2005 and 2025, aggregated at the DOCDB family level.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import polars as pl
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS209_APPLN_IPC,
    TLS224_APPLN_CPC,
)
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sqlalchemy import and_, distinct, func, or_

ENV = "PROD"
IPR_TYPE = "PI"
YEAR_WINDOW = (2004, 2025)
CACHE_DIR = Path("../data/raw")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()

## 2. Core Taxonomy & IPC Reclassification Audit

The unconditional core UAV set comprises classes restricted exclusively to unmanned vehicles by official definition:

- B64U (unmanned aerial vehicle and equipment therefor)
- B64C 39/024 in CPC (of the remote controlled vehicle type, i.e. RPV) mapped to B64C 39/02 in IPC.
- G08G 5/57 and G08G 5/27 (traffic control systems for aircraft, specifically for UAVs)
- G05D 1/228 (command input arrangements located on-board unmanned vehicles)

To account for reclassification lags across national patent officies, we evaluate family coverage across both CPC and IPC.

In [2]:
CORE_PREFIXES = ("B64U%", "G08G5/57%", "G08G5/27%", "G05D1/228%")

def fetch_families(table, col, scheme: str) -> pl.DataFrame:
    clean_sym = func.replace(col, " ", "")
    b64c_prefix = "B64C39/024%" if scheme == "CPC" else "B64C39/02%"
    prefixes = (*CORE_PREFIXES, b64c_prefix)

    q = (
        db.query(TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_auth)
            .select_from(table)
            .join(TLS201_APPLN, table.appln_id == TLS201_APPLN.appln_id)
            .filter(
                TLS201_APPLN.ipr_type == IPR_TYPE,
                TLS201_APPLN.earliest_filing_year.between(*YEAR_WINDOW),
                or_(*(clean_sym.like(p) for p in prefixes)),
            )
            .distinct()
    )

    return pl.from_pandas(patstat.df(q))

df_cpc = fetch_families(TLS224_APPLN_CPC, TLS224_APPLN_CPC.cpc_class_symbol,
                        "CPC")
df_ipc = fetch_families(TLS209_APPLN_IPC, TLS209_APPLN_IPC.ipc_class_symbol,
                        "IPC")

coverage_audit = (
    df_cpc.join(df_ipc, on=["docdb_family_id", "appln_auth"], how="full", coalesce=True)
        .with_columns(
            in_cpc=pl.col("docdb_family_id").is_in(df_cpc["docdb_family_id"]),
            in_ipc=pl.col("docdb_family_id").is_in(df_ipc["docdb_family_id"]),
        )
        .group_by("appln_auth")
        .agg(
            pl.col("docdb_family_id").filter(pl.col("in_cpc")).len().alias("cpc_families"),
            pl.col("docdb_family_id").filter(pl.col("in_ipc") & ~pl.col("in_cpc")).len().alias("ipc_only_families"),
            pl.len().alias("union_families")
        )
        .with_columns(
            (pl.col("ipc_only_families") / pl.col("union_families") * 100)
            .round(2)
            .alias("ipc_only_share_pct")
        )
        .sort("union_families", descending=True)
)

core_family_ids = pl.concat([df_cpc.select("docdb_family_id"),
df_ipc.select("docdb_family_id")]).unique()
print(f"Distinct core UAV docdb families (CPC U IPC): {len(core_family_ids):,}")

coverage_audit.head(10)

Distinct core UAV docdb families (CPC U IPC): 61,967


appln_auth,cpc_families,ipc_only_families,union_families,ipc_only_share_pct
str,u32,u32,u32,f64
"""CN""",26191,7725,33916,22.78
"""US""",14658,346,15004,2.31
"""WO""",10112,589,10701,5.5
"""KR""",8154,1833,9987,18.35
"""JP""",3433,3481,6914,50.35
"""EP""",5355,201,5556,3.62
"""DE""",1266,72,1338,5.38
"""CA""",1210,95,1305,7.28
"""AU""",1203,95,1298,7.32


## 3. Boundary Subclasses Audit

Per Note 2 of the B64U scheme and the G08G 5/57 reclassification warning (which notes pending document transfers from G08G 5/51, 5/54, and 5/56), autonomous vehicle control (G05D 1/00) and air traffic management (G08G 5/00) contain UAV inventions that are not dual-tagged in B64U.

Because these classes also cover manned aviation and ground vehicles, we compute the conditional UAV purity (share of families in each subgroup co-occurring with our core UAV set).

In [3]:
clean_cpc = func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "")
q_boundary = (
    db.query(TLS201_APPLN.docdb_family_id, clean_cpc.label("cpc_symbol"))
        .select_from(TLS224_APPLN_CPC)
        .join(TLS201_APPLN, TLS224_APPLN_CPC.appln_id == TLS201_APPLN.appln_id)
        .filter(
            TLS201_APPLN.ipr_type == IPR_TYPE,
            TLS201_APPLN.earliest_filing_year.between(*YEAR_WINDOW),
            # boundary filters
            or_(clean_cpc.like("G05D1/%"), clean_cpc.like("G08G5/%")),
            ~clean_cpc.like("G08G5/57%"),
            ~clean_cpc.like("G08G5/27%"),
            ~clean_cpc.like("G05D1/228%"),
        )
        .distinct()
)

df_boundary = pl.from_pandas(patstat.df(q_boundary))

subgroup_audit = (
    df_boundary
        .with_columns(
            pl.col("cpc_symbol").str.slice(0, 8).alias("cpc_subgroup"),
            pl.col("docdb_family_id").is_in(core_family_ids["docdb_family_id"]).alias("is_core_uav"),
        )
        .group_by("cpc_subgroup")
        .agg(
            pl.col("docdb_family_id").n_unique().alias("total_families"),
            pl.col("docdb_family_id").filter(pl.col("is_core_uav")).n_unique().alias("core_uav_overlap"),
        )
        # sanity filter
        .filter(pl.col("total_families") >= 150)
        .with_columns(
            (pl.col("core_uav_overlap") / pl.col("total_families") * 100)
            .round(2)
            .alias("uav_purity_pct")
        )
        .sort("uav_purity_pct", descending=True)
)

subgroup_audit

cpc_subgroup,total_families,core_uav_overlap,uav_purity_pct
str,u32,u32,f64
"""G08G5/59""",1280,921,71.95
"""G08G5/55""",5803,3649,62.88
"""G08G5/32""",1645,1000,60.79
"""G08G5/22""",2506,1371,54.71
"""G08G5/58""",631,338,53.57
…,…,…,…
"""G05D1/24""",12138,911,7.51
"""G05D1/64""",8531,463,5.43
"""G05D1/02""",40362,927,2.3


### Decisions for Query Contruction

- **Union CPC and IPC:** Including IPC alongside CPC recovers over 13,000 substantive families missed due to reclassification lags across China (22.8%), Japan (50.4%), and South Korea (18.4%), bringing the unconditional core to 61,967 families.
- **Purity-Thresholded Boundary Pool:** Promoting G08G 5/57 and G08G 5/27 to core reveals that adjacent air-traffic subgroups reach 53%–72% UAV purity (led by geofencing G08G 5/59 at 72.0% and single-aircraft guidance G08G 5/55 at 62.9%).

## 5. Boundary Pool & Semantic Query Calibration

To retrieve un-reclassified UAV inventions inside the retained boundary subgroups (purity ≥ 20%), we construct a balanced benchmark pool of 2,000 families with English titles and abstracts (1,000 core-verified UAV families and 1,000 unlabeled boundary families).

Using a stratified evaluation split, we benchmark deterministic keywords against two semantic classifiers built on Patent SBERT embeddings (graziasveva93/sbert-adapt-ub Ascione & Sterzi, 2024):

1. **Constrative Centroid:** Cosine similarity to the UAV prototype minus 0.5* similarity to the unlabeled boundary prototype.
2. **Logistic Regression Probe:** A linear classifier that down-weights shared aviation control dimensions and isolates UAV-specific latent features, with decision thresholds calibrated via 5-fold out-of-fold predictions to maximize F0.5.

In [4]:
POOL_FILE = (CACHE_DIR / f"boundary_pool_p20_n2000.parquet")
valid_subgroups = subgroup_audit.filter(pl.col("uav_purity_pct") >= 20.0)["cpc_subgroup"]
valid_subgroups

cpc_subgroup
str
"""G08G5/59"""
"""G08G5/55"""
"""G08G5/32"""
"""G08G5/22"""
"""G08G5/58"""
…
"""G05D1/10"""
"""G08G5/21"""
"""G05D1/22"""


In [5]:
if POOL_FILE.exists():
    df_pool = pl.read_parquet(POOL_FILE)
else:
    boundary_fams = (
        df_boundary
            .with_columns(pl.col("cpc_symbol").str.slice(0, 8).alias("cpc_subgroup"))
            .filter(pl.col("cpc_subgroup").is_in(valid_subgroups))
            .select("docdb_family_id")
            .unique()
            .with_columns(pl.col("docdb_family_id").is_in(core_family_ids["docdb_family_id"]).alias("is_uav"))
    )

    # extracts 2,000 positive samples and 2,000 negative samples
    # note: we only need 1,000 each, but we have to filter them, so this is a "safety check"
    candidates = pl.concat(
        [
            boundary_fams.filter(pl.col("is_uav")).sample(n=2000, seed=42),
            boundary_fams.filter(~pl.col("is_uav")).sample(n=2000, seed=42)
        ]
    )

    q_text = (
        db.query(
            TLS201_APPLN.docdb_family_id,
            TLS202_APPLN_TITLE.appln_title.label("title"),
            TLS203_APPLN_ABSTR.appln_abstract.label("abstract")
        )
            .select_from(TLS201_APPLN)
            .join(TLS202_APPLN_TITLE, TLS202_APPLN_TITLE.appln_id == TLS201_APPLN.appln_id)
            .join(TLS203_APPLN_ABSTR, TLS203_APPLN_ABSTR.appln_id == TLS201_APPLN.appln_id)
            .filter(
                TLS201_APPLN.docdb_family_id.in_(candidates["docdb_family_id"].to_list()),
                TLS202_APPLN_TITLE.appln_title_lg == "en",
                TLS203_APPLN_ABSTR.appln_abstract_lg == "en"
            )
    )

    df_raw_text = (
        pl.from_pandas(patstat.df(q_text))
            .unique(subset=["docdb_family_id"], keep="first")
            .join(candidates, on="docdb_family_id", how="inner")
            .with_columns(pl.concat_str([pl.col("title"), pl.lit(". "), pl.col("abstract")]).alias("text"))
    )

    df_pool = pl.concat(
        [
            df_raw_text.filter(pl.col("is_uav")).sample(n=1000, seed=42),
            df_raw_text.filter(~pl.col("is_uav")).sample(n=1000, seed=42),
        ]
    )
    df_pool.write_parquet(POOL_FILE)

print(f"Boundary pool loaded: {len(df_pool):,} families across {len(valid_subgroups)} subgroups")
df_pool.group_by("is_uav").len()

Boundary pool loaded: 2,000 families across 32 subgroups


is_uav,len
bool,u32
true,1000
false,1000


In [6]:
POS_KEYWORDS = [
    r"\buav\b",
    r"\buavs\b",
    r"\buas\b",
    r"\brpas\b",
    r"\bunmanned aerial\b",
    r"\buncrewed aerial\b",
    r"\bremotely piloted\b",
    r"\bdrone\b",
    r"\bdrones\b",
    r"\bquadrotor\b",
    r"\bquadcopter\b",
    r"\bmultirotor\b",
    r"\bmulticopter\b",
    r"\bhexacopter\b",
    r"\boctocopter\b",
    r"\bevtol\b",
    r"\brotorcraft\b",
    r"\bflying robot\b",
]

NEG_KEYWORDS = [
    r"\bautonomous driving\b",
    r"\broad vehicle\b",
    r"\blawnmower\b",
    r"\bsubmersible\b",
    r"\bunderwater vehicle\b",
    r"\bauv\b",
    r"\busv\b",
]

Note: These are the sets of positive and negative keywords that we will be using. Some were generated using Gemini Argon 4, while others were selected following a review of relevant academic papers.

In [7]:
def match_keywords(
    texts: pl.Series,
    pos_kw: list[str],
    neg_kw: list[str]
) -> np.ndarray:
    positive = "(?i)(" + "|".join(pos_kw) + ")"
    negative = "(?i)(" + "|".join(neg_kw) + ")"
    return (texts.str.contains(positive) & ~texts.str.contains(negative)).to_numpy()

# Contrastive semantic score
# cosine similarity to the UAV centroid minus 0.5* similarity to the non-UAV background centroid.
def compute_sbert_score(emb: np.ndarray) -> np.ndarray:
    return (emb @ uav_proto.T).ravel() - 0.5 * (emb @ bg_proto.T).ravel()

def compute_f05(pred: np.ndarray, target: np.ndarray) -> float:
    tp = (pred & target).sum()
    prec = tp / max(pred.sum(), 1)
    rec = tp / max(target.sum(), 1)
    return (1.25 * prec * rec) / max(0.25 * prec + rec, 1e-9)

def evaluate(name: str, p_eval: np.ndarray, t_eval: np.ndarray) -> dict:
    tp = int((p_eval & t_eval).sum())
    prec = tp / max(int(p_eval.sum()), 1)
    rec = tp / max(int(t_eval.sum()), 1)
    f05 = (1.25 * prec * rec) / max(0.25 * prec + rec, 1e-9)
    f1 = (2 * prec * rec) / max(prec + rec, 1e-9)
    return {
        "query": name,
        "selected": int(p_eval.sum()),
        "precision": round(prec, 3),
        "recall": round(rec, 3),
        "F0.5": round(f05, 3),
        "F1": round(f1, 3),
    }

In [8]:
# deterministic baseline
by_kw = match_keywords(df_pool["text"], POS_KEYWORDS, NEG_KEYWORDS)

# ground truth
truth = df_pool["is_uav"].to_numpy()

patent_sbert = SentenceTransformer("graziasveva93/sbert-adapt-ub")
embeddings = patent_sbert.encode(
    df_pool["text"].to_list(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

# shapes: emb (400, 384) / (1600, 384) | y & kw are True/False vectors
emb_seed, emb_eval, y_seed, y_eval, kw_seed, kw_eval = train_test_split(
    embeddings,
    truth,
    by_kw,
    # since sbert-adapt-ub is a pre-trained model
    # we shouldnt need a large training set (20%)
    train_size=.2,
    stratify=truth,
    random_state=42
)

uav_proto = normalize(emb_seed[y_seed].mean(axis=0, keepdims=True))
bg_proto = normalize(emb_seed[~y_seed].mean(axis=0, keepdims=True))

# sbert_score measures how closely a patent aligns with the UAV
# semantic centroid relative to the non-UAV background centroid
seed_score = compute_sbert_score(emb_seed)
eval_score = compute_sbert_score(emb_eval)

# candidate tau (decision threshold)
grid = np.linspace(np.percentile(seed_score, 10), np.percentile(seed_score, 95), 200)
# the 'grid' value that get the highest f0.5-score
best_tau = max(
    grid,
    key=lambda t: compute_f05(seed_score >= t, y_seed)
)
sbert_eval = eval_score >= best_tau

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

# --- New Try: using a Logistic Regression model to fine-tune the tau ---
lr = LogisticRegression(C=0.2, max_iter=1000, random_state=42)

seed_prob_lr = cross_val_predict(
    lr,
    emb_seed, y_seed,
    cv=5,
    method="predict_proba"
)[:, 1]
best_tau_lr = max(
    np.linspace(0.1, 0.9, 200),
    key=lambda t: compute_f05(seed_prob_lr >= t, y_seed)
)

lr.fit(emb_seed, y_seed)
eval_prob_lr = lr.predict_proba(emb_eval)[:, 1]
sbert_lr_eval = eval_prob_lr >= best_tau_lr

In [10]:
pl.DataFrame([
    evaluate("Keywords", kw_eval, y_eval),
    evaluate("Patent-SBERT", sbert_eval, y_eval),
    evaluate("Keywords OR SBERT", kw_eval | sbert_eval, y_eval),
    evaluate("Keywords AND SBERT", kw_eval & sbert_eval, y_eval),
    evaluate("SBERT + LogReg", sbert_lr_eval, y_eval),
    evaluate("Keywords OR (SBERT+LogReg)", kw_eval | sbert_lr_eval, y_eval),
])

query,selected,precision,recall,F0.5,F1
str,i64,f64,f64,f64,f64
"""Keywords""",669,0.75,0.627,0.722,0.683
"""Patent-SBERT""",667,0.672,0.56,0.646,0.611
"""Keywords OR SBERT""",883,0.691,0.762,0.704,0.725
"""Keywords AND SBERT""",453,0.751,0.425,0.651,0.543
"""SBERT + LogReg""",821,0.705,0.724,0.709,0.714
"""Keywords OR (SBERT+LogReg)""",953,0.69,0.823,0.713,0.751


## 6. Semantic Term Discovery & Query Refinement

After running multiple tests across different pools, the results showed that the deterministic baseline (keywords) slightly outperforms all the alternatives.

We use SBERT + LogReg as a semantic discovery tool rather than a direct classifier. Specifically, we isolate true UAV patents captured by SBERT + LogReg but missed by Keywords, extract their most distinctive unigrams and bigrams via differential TF-IDF (filtering out patent boilerplate stopwords), and manually integrate high-precision synonyms (if needed) into POS_KEYWORDS before final validation on an unseen pool.

In [11]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

PATENT_STOP_WORDS = {
    "said", "wherein", "whereby", "thereof", "therein", "comprising",
    "comprises",
    "comprise", "including", "includes", "plurality", "least", "configured",
    "provided", "providing", "present", "invention", "disclosed",
    "disclosure",
    "embodiment", "embodiments", "method", "methods", "device", "devices",
    "apparatus", "system", "systems", "unit", "first", "second", "third",
    "based",
}

_, df_eval = train_test_split(
    df_pool,
    train_size=0.2,
    stratify=truth,
    random_state=42
)

# true UAV, only by SBERT + LogReg
only_semantic = sbert_lr_eval & ~kw_eval & y_eval
# true not-UAV
true_neg = ~sbert_lr_eval & ~kw_eval & ~y_eval

vec = TfidfVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS | PATENT_STOP_WORDS),
    ngram_range=(1, 2),
    min_df=5,
    sublinear_tf=True,
)
X_eval_tfidf = vec.fit_transform(df_eval["text"].to_list())

diff_tfidf = np.asarray(
    X_eval_tfidf[only_semantic].mean(axis=0) -
    X_eval_tfidf[true_neg].mean(axis=0)
).ravel()
top_idx = diff_tfidf.argsort()[::-1][:15]

discovered_terms = pl.DataFrame({
    "discovered_ngram": vec.get_feature_names_out()[top_idx],
    "diff_tfidf_lift": diff_tfidf[top_idx].round(4),
})

print(f"True UAVs caught by SBERT+LogReg but missed by Keywords: {len(df_eval.filter(pl.Series(only_semantic)))}")
with pl.Config(tbl_rows=15):
    display(discovered_terms)

True UAVs caught by SBERT+LogReg but missed by Keywords: 156


discovered_ngram,diff_tfidf_lift
str,f64
"""aircraft""",0.0314
"""flying""",0.0262
"""aerial""",0.025
"""flight""",0.0237
"""communication""",0.0215
"""unmanned""",0.0195
"""unmanned aircraft""",0.0175
"""image""",0.0172
"""aerial vehicle""",0.0169


In [12]:
REFINED_POS_KEYWORDS = [
    *POS_KEYWORDS,
    r"\bunmanned aircraft\b",
    r"\bunmanned air\b",
    r"\bunmanned flight\b",
    r"\bunmanned flying\b",
    r"\bflying body\b",
    r"\bflying bodies\b",
    r"\bflying vehicle\b",
    r"\bflight vehicle\b",
    r"\baerial vehicle\b",
    r"\baerial vehicles\b",
    r"\baerial platform\b",
    r"\bvtol\b",
    r"\bmav\b",
    r"\bmicro air\b",
]

kw_refined_eval = match_keywords(df_eval["text"], REFINED_POS_KEYWORDS,
                                 NEG_KEYWORDS)

pl.DataFrame([
    evaluate("Keywords (Baseline)", kw_eval, y_eval),
    evaluate("Keywords (Refined)", kw_refined_eval, y_eval),
    evaluate("SBERT + LogReg", sbert_lr_eval, y_eval),
])

query,selected,precision,recall,F0.5,F1
str,i64,f64,f64,f64,f64
"""Keywords (Baseline)""",669,0.75,0.627,0.722,0.683
"""Keywords (Refined)""",746,0.76,0.709,0.749,0.734
"""SBERT + LogReg""",821,0.705,0.724,0.709,0.714


## 7. Family Validation Phase (n = 100)

The final query is the union of two **disjoint** sets:

1. **Core Classification:** families in `core_family_ids` (the CPC ∪ IPC unconditional core).
2. **Boundary Rescue:** non-core families in the CPC boundary subgroups with `uav_purity_pct >= 20%`, whose English title **or** English abstract matches `REFINED_POS_KEYWORDS` and does not match `NEG_KEYWORDS`.

We draw a stratified random sample of 100 distinct DOCDB families. The two strata are the two components of the final query. A minimum of 20 families is allocated to each non-empty stratum; the remaining sample is allocated approximately in proportion to the remaining population sizes. The estimator uses the corresponding population weights, so disproportionate allocation does not bias the estimated precision.

Each family receives one primary annotation. A random subset of 20 families is independently annotated by a second person to measure inter-annotator agreement. The second annotation does **not** increase the sample size: the validation sample remains 100 distinct families.

**Important:** this section estimates the precision of the final query. It does not estimate global recall. Estimating recall requires a separately defined and sampled population of candidate families outside the final query.

### 7.1. Annotation rubric

For every sampled family, inspect the English title and abstract provided below.

- `is_drone = 1`: the claimed invention concerns an unmanned aircraft (UAV/UAS/RPAS, multirotor, fixed-wing or VTOL), or an essential system/component in the defined drone ecosystem (for example flight control, payload, launch/recovery, energy or swarm systems).
- `is_drone = 0`: the drone is only an incidental example, the invention concerns a non-aerial vehicle (for example a ground robot, surface/underwater vessel, missile or manned aircraft), or the invention is unrelated.
- `is_drone = 2`: the family cannot be classified from the supplied title and abstract.

Only when `is_drone = 1`, assign `use = civil | military | dual | unclear`. The use must be supported by the patent text, not inferred from the applicant's identity. If the supplied English text does not establish the use, choose `unclear`.

The legacy annotation rubric also included anti-drone systems. Keep them in scope only if the project's agreed sector definition includes counter-UAS technologies; otherwise classify them as outside the core drone sector and explain the decision in `comment`.

The source stratum and query route are kept out of the annotator files to reduce expectation bias.

In [20]:
# Imports used by the validation phase. This section does not require sentence-transformers.
import re
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from scipy.stats import beta, norm
from sqlalchemy import or_

# Reproducibility and annotation settings
N_SAMPLE = 100
MIN_PER_STRATUM = 20
N_DOUBLE = 20
RANDOM_SEED = 42
ANNOTATORS = ["A", "B", "C"]  # Replace with the annotators' identifiers

VALIDATION_DIR = Path("../data/uav_core_boundary_validation")
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# The following objects must already exist from Sections 1–6:
# db, patstat, IPR_TYPE, YEAR_WINDOW,
# TLS201_APPLN, TLS202_APPLN_TITLE, TLS203_APPLN_ABSTR,
# core_family_ids, df_boundary, valid_subgroups,
# REFINED_POS_KEYWORDS, NEG_KEYWORDS

In [21]:
# 7.2. Build the two disjoint populations of the final query


# Set 1: unconditional core families (CPC union IPC)
core_ids = set(
    int(x) for x in core_family_ids["docdb_family_id"].drop_nulls().to_list()
)

# Candidate families in boundary CPC subgroups passing the >=20% UAV-purity threshold
boundary_candidates = (
    df_boundary
    .with_columns(
        pl.col("cpc_symbol").str.slice(0, 8).alias("cpc_subgroup")
    )
    .filter(pl.col("cpc_subgroup").is_in(valid_subgroups))
    .select("docdb_family_id")
    .unique()
)

# Set 2 must exclude all families already present in the core.
boundary_candidate_ids = [
    int(x) for x in boundary_candidates["docdb_family_id"].to_list()
    if int(x) not in core_ids
]

if not core_ids:
    raise RuntimeError("core_family_ids is empty. Run the core CPC/IPC extraction first.")
if not boundary_candidate_ids:
    raise RuntimeError("No non-core families were found in the retained boundary subgroups.")

# Build an English-text record for each family. Text is fetched in batches to avoid
# constructing a single very large SQL IN clause.
def _first_nonempty(values):
    for value in values:
        if pd.notna(value) and str(value).strip():
            return str(value)
    return ""


def fetch_family_text(family_ids, batch_size=3000):
    family_ids = list(dict.fromkeys(int(x) for x in family_ids if pd.notna(x)))
    frames = []

    for start in range(0, len(family_ids), batch_size):
        batch = family_ids[start:start + batch_size]
        q = (
            db.query(
                TLS201_APPLN.docdb_family_id.label("docdb_family_id"),
                TLS201_APPLN.earliest_filing_year.label("priority_year"),
                TLS202_APPLN_TITLE.appln_title.label("title"),
                TLS202_APPLN_TITLE.appln_title_lg.label("title_lang"),
                TLS203_APPLN_ABSTR.appln_abstract.label("abstract"),
                TLS203_APPLN_ABSTR.appln_abstract_lg.label("abstract_lang"),
            )
            .select_from(TLS201_APPLN)
            .outerjoin(
                TLS202_APPLN_TITLE,
                TLS202_APPLN_TITLE.appln_id == TLS201_APPLN.appln_id,
            )
            .outerjoin(
                TLS203_APPLN_ABSTR,
                TLS203_APPLN_ABSTR.appln_id == TLS201_APPLN.appln_id,
            )
            .filter(
                TLS201_APPLN.docdb_family_id.in_(batch),
                TLS201_APPLN.ipr_type == IPR_TYPE,
                TLS201_APPLN.earliest_filing_year.between(*YEAR_WINDOW),
            )
        )
        chunk = patstat.df(q)
        if chunk is not None and not chunk.empty:
            frames.append(chunk)

    requested = pd.DataFrame({"docdb_family_id": family_ids})
    if not frames:
        return requested.assign(priority_year=np.nan, title="", abstract="")

    raw = pd.concat(frames, ignore_index=True)
    raw["title_en"] = raw["title"].where(raw["title_lang"].fillna("").str.lower().eq("en"))
    raw["abstract_en"] = raw["abstract"].where(raw["abstract_lang"].fillna("").str.lower().eq("en"))

    grouped = (
        raw.groupby("docdb_family_id", as_index=False)
        .agg(
            priority_year=("priority_year", "min"),
            title=("title_en", _first_nonempty),
            abstract=("abstract_en", _first_nonempty),
        )
    )
    result = requested.merge(grouped, on="docdb_family_id", how="left")
    result["title"] = result["title"].fillna("")
    result["abstract"] = result["abstract"].fillna("")
    return result


# Retrieve English titles/abstracts for boundary candidates, then apply the exact
# positive and negative regex lists calibrated in Section 6.
boundary_text = fetch_family_text(boundary_candidate_ids)
positive_pattern = re.compile("|".join(f"(?:{p})" for p in REFINED_POS_KEYWORDS), re.IGNORECASE)
negative_pattern = re.compile("|".join(f"(?:{p})" for p in NEG_KEYWORDS), re.IGNORECASE)

boundary_search_text = boundary_text["title"].fillna("") + ". " + boundary_text["abstract"].fillna("")
has_positive = boundary_search_text.str.contains(positive_pattern, regex=True, na=False)
has_negative = boundary_search_text.str.contains(negative_pattern, regex=True, na=False)

boundary_rescue = boundary_text.loc[has_positive & ~has_negative].copy()
boundary_rescue = boundary_rescue.drop_duplicates("docdb_family_id").reset_index(drop=True)

# Check disjointness and report the final query population sizes.
boundary_ids = set(int(x) for x in boundary_rescue["docdb_family_id"].tolist())
if core_ids.intersection(boundary_ids):
    raise RuntimeError("Core Classification and Boundary Rescue are not disjoint.")

populations = {
    "Core Classification": len(core_ids),
    "Boundary Rescue": len(boundary_ids),
}
N_QUERY = sum(populations.values())
if N_QUERY == 0:
    raise RuntimeError("The final query returned no families.")

print("Final query populations:")
print(pd.Series(populations, name="families"))
print(f"Total distinct DOCDB families: {N_QUERY:,}")

# Export the full query-family list for the separate trends-analysis notebook.
query_family_frame = pd.concat(
    [
        pd.DataFrame({"docdb_family_id": sorted(core_ids), "query_source": "core_classification"}),
        pd.DataFrame({"docdb_family_id": sorted(boundary_ids), "query_source": "boundary_rescue"}),
    ],
    ignore_index=True,
).drop_duplicates("docdb_family_id")
query_family_frame.to_csv(VALIDATION_DIR / "uav_final_query_families.csv", index=False)

Final query populations:
Core Classification    61967
Boundary Rescue         9266
Name: families, dtype: int64
Total distinct DOCDB families: 71,233


In [22]:
# 7.3. Stratified random sample of 100 distinct families

def allocate_stratified_sample(populations, n_total=100, min_per_stratum=20):
    """Allocate n_total across the two non-empty strata, with a minimum per stratum."""
    active = {name: int(size) for name, size in populations.items() if int(size) > 0}
    if not active:
        raise ValueError("No non-empty stratum is available.")
    if len(active) > n_total:
        raise ValueError("Sample size is too small to cover every non-empty stratum.")

    target = min(int(n_total), sum(active.values()))
    allocation = {name: min(size, int(min_per_stratum)) for name, size in active.items()}
    remaining = target - sum(allocation.values())

    # Distribute the remaining sample approximately in proportion to each stratum's
    # remaining population. The largest-remainder step makes the allocation sum exact.
    while remaining > 0:
        capacity = {name: active[name] - allocation[name] for name in active}
        capacity_total = sum(capacity.values())
        if capacity_total <= 0:
            break

        quotas = {name: remaining * capacity[name] / capacity_total for name in active}
        floors = {
            name: min(capacity[name], int(np.floor(quotas[name])))
            for name in active
        }
        added = sum(floors.values())
        for name, value in floors.items():
            allocation[name] += value
        remaining -= added

        if remaining > 0:
            order = sorted(
                active,
                key=lambda name: quotas[name] - floors[name],
                reverse=True,
            )
            progressed = False
            for name in order:
                if remaining == 0:
                    break
                if allocation[name] < active[name]:
                    allocation[name] += 1
                    remaining -= 1
                    progressed = True
            if not progressed and added == 0:
                break

    if sum(allocation.values()) != target:
        raise RuntimeError("Could not allocate the requested stratified sample size.")
    return allocation


sample_sizes = allocate_stratified_sample(
    populations,
    n_total=N_SAMPLE,
    min_per_stratum=MIN_PER_STRATUM,
)
print("Sample allocation:")
print(pd.Series(sample_sizes, name="sample_n"))

core_pool = pd.DataFrame({"docdb_family_id": sorted(core_ids)})
boundary_pool = boundary_rescue.copy()
stratum_pools = {
    "Core Classification": core_pool,
    "Boundary Rescue": boundary_pool,
}

sample_parts = []
for i, (stratum, pool) in enumerate(stratum_pools.items()):
    n_h = sample_sizes.get(stratum, 0)
    if n_h == 0:
        continue
    selected = pool.sample(n=n_h, replace=False, random_state=RANDOM_SEED + i)
    selected_ids = selected["docdb_family_id"].astype(int).tolist()

    if stratum == "Core Classification":
        details = fetch_family_text(selected_ids)
    else:
        details = selected[["docdb_family_id", "priority_year", "title", "abstract"]].copy()

    details["stratum"] = stratum
    details["N_h"] = populations[stratum]
    details["n_h"] = n_h
    sample_parts.append(details)

validation_sample = pd.concat(sample_parts, ignore_index=True)
validation_sample["has_english_text"] = (
    validation_sample["title"].fillna("").str.strip().ne("")
    | validation_sample["abstract"].fillna("").str.strip().ne("")
)
validation_sample = validation_sample.drop_duplicates("docdb_family_id")

if len(validation_sample) != min(N_SAMPLE, N_QUERY):
    raise RuntimeError(
        f"Expected {min(N_SAMPLE, N_QUERY)} unique families, found {len(validation_sample)}."
    )

# Preserve the sampling frame and its stratum information for the analysis.
validation_sample.to_csv(VALIDATION_DIR / "validation_sample_master.csv", index=False)
print(f"Unique families selected for manual validation: {len(validation_sample):,}")
print("Families without English title and abstract:", int((~validation_sample["has_english_text"]).sum()))

Sample allocation:
Core Classification    72
Boundary Rescue        28
Name: sample_n, dtype: int64
Unique families selected for manual validation: 100
Families without English title and abstract: 1


### 7.4. Create blinded annotation files

Each of the 100 sampled families receives one primary annotation. `N_DOUBLE = 20` families are randomly selected for a second independent annotation. The code creates one CSV per annotator. The `stratum` and query source are not included in the files sent to annotators.

The annotation files contain blank `is_drone`, `use` and `comment` columns. Annotators should fill `is_drone` with `0`, `1` or `2`, and `use` with `civil`, `military`, `dual` or `unclear` when `is_drone = 1`.

In [23]:
# Randomly assign one primary annotator to each unique family.
primary_assignments = validation_sample.sample(
    frac=1, random_state=RANDOM_SEED
).reset_index(drop=True).copy()
primary_assignments["annotator"] = [
    ANNOTATORS[i % len(ANNOTATORS)] for i in range(len(primary_assignments))
]
primary_assignments["annotation_role"] = "primary"
primary_assignments["annotation_id"] = [
    f"P{i + 1:03d}" for i in range(len(primary_assignments))
]

# Select N_DOUBLE different families for independent second annotation.
if N_DOUBLE > len(primary_assignments):
    raise ValueError("N_DOUBLE cannot exceed the number of unique sampled families.")

double_assignments = primary_assignments.sample(
    n=N_DOUBLE, random_state=RANDOM_SEED + 100
).copy().reset_index(drop=True)

annotator_next = {
    ANNOTATORS[i]: ANNOTATORS[(i + 1) % len(ANNOTATORS)]
    for i in range(len(ANNOTATORS))
}
double_assignments["annotator"] = double_assignments["annotator"].map(annotator_next)
double_assignments["annotation_role"] = "double"
double_assignments["annotation_id"] = [
    f"D{i + 1:03d}" for i in range(len(double_assignments))
]

annotation_master = pd.concat(
    [primary_assignments, double_assignments], ignore_index=True
)
annotation_master.to_csv(
    VALIDATION_DIR / "validation_assignments_master.csv", index=False
)

# Separate blinded files: do not expose stratum, N_h, n_h or query_source.
blind_columns = [
    "annotation_id",
    "docdb_family_id",
    "priority_year",
    "title",
    "abstract",
    "has_english_text",
    "annotator",
]
for annotator in ANNOTATORS:
    assigned = annotation_master.loc[
        annotation_master["annotator"] == annotator, blind_columns
    ].copy()
    assigned["is_drone"] = ""
    assigned["use"] = ""
    assigned["comment"] = ""
    assigned.to_csv(
        VALIDATION_DIR / f"uav_core_boundary_validation_to_{annotator}.csv", index=False
    )

print("Annotation files created:")
for annotator in ANNOTATORS:
    print(f" - {VALIDATION_DIR / f'uav_core_boundary_validation_to_{annotator}.csv'}")
print(f"Total annotation rows (100 primary + {N_DOUBLE} double): {len(annotation_master)}")

Annotation files created:
 - ../data/uav_core_boundary_validation/uav_core_boundary_validation_to_A.csv
 - ../data/uav_core_boundary_validation/uav_core_boundary_validation_to_B.csv
 - ../data/uav_core_boundary_validation/uav_core_boundary_validation_to_C.csv
Total annotation rows (100 primary + 20 double): 120


### 7.5. Estimate query precision after manual annotation

After the annotators have completed their CSV files, place the completed files back in `VALIDATION_DIR` under the same names and run the next cell.

The primary outcome is conservative: `is_drone = 1` is counted as a confirmed true positive; both `0` and `2` are counted as not confirmed. The share of `2` labels is reported separately, so uncertain records are not silently removed from the denominator.

For each stratum, the estimate is `p_h = k_h / n_h`. The overall stratified estimator is:

\[
\widehat{P}_{st} = \sum_h W_h\widehat{p}_h, \qquad W_h = N_h / N.
\]

The Cochran variance estimate for stratified simple random sampling without replacement is:

\[
\widehat{V}(\widehat{P}_{st}) = \sum_h W_h^2(1-n_h/N_h) s_h^2/n_h.
\]

The output includes the Cochran normal interval, a conservative weighted Wilson interval (Bonferroni-adjusted across the two strata), and a stratified Jeffreys interval obtained by combining independent Jeffreys beta posteriors. The Jeffreys interval is Bayesian (credible), not a frequentist confidence interval.

In [24]:
# 7.6. Read completed annotations and estimate stratified precision
from sklearn.metrics import cohen_kappa_score

annotation_paths = [
    VALIDATION_DIR / f"uav_core_boundary_validation_to_{annotator}.csv"
    for annotator in ANNOTATORS
]
missing_paths = [path for path in annotation_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing annotation files: {missing_paths}")

annotation_frames = [pd.read_csv(path) for path in annotation_paths]
annotations = pd.concat(annotation_frames, ignore_index=True)

# Guard against accidentally reusing CSVs from the older validation notebook.
required_annotation_columns = {
    "annotation_id", "docdb_family_id", "annotator", "is_drone", "use", "comment"
}
missing_columns = required_annotation_columns.difference(annotations.columns)
if missing_columns:
    raise ValueError(
        "These are not annotation files generated by this Core Classification / "
        "Boundary Rescue validation run. Missing columns: "
        f"{sorted(missing_columns)}. Regenerate the files in {VALIDATION_DIR} "
        "and annotate those files; do not reuse the older validation_to_A/B/C.csv files."
    )

annotations["annotation_id"] = annotations["annotation_id"].astype(str).str.strip()
annotations["annotator"] = annotations["annotator"].astype(str).str.strip()
annotations["docdb_family_id"] = pd.to_numeric(
    annotations["docdb_family_id"], errors="coerce"
)
if annotations["docdb_family_id"].isna().any():
    raise ValueError("Some annotation rows have a missing or invalid docdb_family_id.")
annotations["docdb_family_id"] = annotations["docdb_family_id"].astype("int64")

expected_assignments = annotation_master[
    ["annotation_id", "docdb_family_id", "annotator"]
].copy()
expected_assignments["annotation_id"] = expected_assignments["annotation_id"].astype(str)
expected_assignments["annotator"] = expected_assignments["annotator"].astype(str)
expected_assignments["docdb_family_id"] = expected_assignments["docdb_family_id"].astype("int64")

if annotations["annotation_id"].duplicated().any():
    raise ValueError("Duplicate annotation_id found across the uploaded annotation files.")
if set(annotations["annotation_id"]) != set(expected_assignments["annotation_id"]):
    missing_ids = sorted(set(expected_assignments["annotation_id"]) - set(annotations["annotation_id"]))
    extra_ids = sorted(set(annotations["annotation_id"]) - set(expected_assignments["annotation_id"]))
    raise ValueError(
        "The uploaded annotation files do not match the 100-family sample generated "
        "in this notebook. "
        f"Missing annotation IDs (first 10): {missing_ids[:10]}; "
        f"unexpected IDs (first 10): {extra_ids[:10]}. "
        "Use the newly generated uav_core_boundary_validation_to_A/B/C.csv files."
    )

check_assignments = annotations[
    ["annotation_id", "docdb_family_id", "annotator"]
].merge(
    expected_assignments,
    on="annotation_id",
    how="left",
    suffixes=("_uploaded", "_expected"),
    validate="one_to_one",
)
if (
    (check_assignments["docdb_family_id_uploaded"] != check_assignments["docdb_family_id_expected"]).any()
    or (check_assignments["annotator_uploaded"] != check_assignments["annotator_expected"]).any()
):
    raise ValueError(
        "At least one annotation_id has been associated with the wrong family or annotator. "
        "Please restore the original generated CSV files before analysis."
    )

annotations["is_drone"] = pd.to_numeric(annotations["is_drone"], errors="coerce")

if annotations["is_drone"].isna().any():
    raise ValueError("Some is_drone labels are empty or non-numeric. Complete all annotations first.")
if not annotations["is_drone"].isin([0, 1, 2]).all():
    raise ValueError("is_drone must contain only 0, 1 or 2.")

# Validate the use labels for positive drone families.
valid_uses = {"civil", "military", "dual", "unclear"}
positive_rows = annotations[annotations["is_drone"] == 1]
invalid_uses = positive_rows.loc[
    ~positive_rows["use"].fillna("").str.lower().isin(valid_uses)
]
if not invalid_uses.empty:
    raise ValueError(
        "For every is_drone=1 row, use must be civil, military, dual or unclear."
    )

# Inter-annotator agreement on the 20 double-annotated families.
double_ids = set(
    annotation_master.loc[
        annotation_master["annotation_role"] == "double", "docdb_family_id"
    ].astype(int)
)
double_rows = annotations[
    annotations["docdb_family_id"].astype(int).isin(double_ids)
].copy()
pair_records = []
for family_id, group in double_rows.groupby("docdb_family_id"):
    group = group.dropna(subset=["is_drone"]).drop_duplicates("annotator")
    if len(group) == 2:
        pair_records.append({
            "docdb_family_id": family_id,
            "label_1": int(group.iloc[0]["is_drone"]),
            "label_2": int(group.iloc[1]["is_drone"]),
        })
pairs = pd.DataFrame(pair_records)
if len(pairs) >= 2:
    kappa_3class = cohen_kappa_score(
        pairs["label_1"], pairs["label_2"], labels=[0, 1, 2]
    )
else:
    kappa_3class = np.nan

# Conservative family-level consensus: when double annotations disagree,
# mark the family as uncertain (2) rather than choosing an annotator arbitrarily.
def resolve_family_annotations(group):
    labels = group["is_drone"].dropna().astype(int).tolist()
    if not labels:
        return pd.Series({"is_drone_final": np.nan, "use_final": "unclear"})
    final_label = labels[0] if len(set(labels)) == 1 else 2
    uses = (
        group.loc[group["is_drone"] == 1, "use"]
        .dropna().astype(str).str.lower().tolist()
    )
    use_final = uses[0] if uses and len(set(uses)) == 1 else "unclear"
    return pd.Series({"is_drone_final": final_label, "use_final": use_final})

family_label_rows = []
for family_id, group in annotations.groupby("docdb_family_id"):
    resolved = resolve_family_annotations(group)
    family_label_rows.append({"docdb_family_id": family_id, **resolved.to_dict()})
family_labels = pd.DataFrame(family_label_rows)
validated_sample = validation_sample.merge(
    family_labels, on="docdb_family_id", how="left", validate="one_to_one"
)

# Cochran stratified estimator and stratum-level intervals.
alpha = 0.05
n_strata = sum(1 for name, size in populations.items() if size > 0)
alpha_per_stratum = alpha / n_strata  # Bonferroni correction for the combined Wilson interval
z_cochran = norm.ppf(1 - alpha / 2)
z_wilson = norm.ppf(1 - alpha_per_stratum / 2)
N_total = sum(populations.values())


def wilson_interval(k, n, z):
    if n == 0:
        return (np.nan, np.nan)
    phat = k / n
    denominator = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denominator
    half_width = (
        z * np.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2))
        / denominator
    )
    return (max(0.0, center - half_width), min(1.0, center + half_width))


stratum_rows = []
weighted_estimate = 0.0
cochran_variance = 0.0
combined_wilson_low = 0.0
combined_wilson_high = 0.0
weighted_uncertainty = 0.0

for stratum, N_h in populations.items():
    if N_h == 0:
        continue
    group = validated_sample.loc[validated_sample["stratum"] == stratum]
    n_h = len(group)
    if n_h == 0:
        raise RuntimeError(f"No annotations found for non-empty stratum: {stratum}")

    # Conservative definition: only an explicit label 1 is a confirmed positive.
    k_h = int((group["is_drone_final"] == 1).sum())
    unsure_h = int((group["is_drone_final"] == 2).sum())
    p_h = k_h / n_h
    W_h = N_h / N_total

    weighted_estimate += W_h * p_h
    weighted_uncertainty += W_h * (unsure_h / n_h)

    if n_h > 1 and n_h < N_h:
        s2_h = (n_h / (n_h - 1)) * p_h * (1 - p_h)
        cochran_variance += (W_h ** 2) * (1 - n_h / N_h) * s2_h / n_h
    elif n_h == N_h:
        # Census of this stratum: no sampling variance for this stratum.
        s2_h = 0.0
    else:
        raise RuntimeError(f"A stratum has n_h={n_h}; at least 2 are needed unless it is a census.")

    # If the stratum is a census, its finite-population precision is known exactly.
    if n_h == N_h:
        wilson95_low = wilson95_high = p_h
        jeffreys95_low = jeffreys95_high = p_h
        wilson_adj_low = wilson_adj_high = p_h
    else:
        # Nominal 95% intervals for reporting each stratum separately.
        wilson95_low, wilson95_high = wilson_interval(k_h, n_h, norm.ppf(0.975))
        jeffreys95_low = beta.ppf(0.025, k_h + 0.5, n_h - k_h + 0.5)
        jeffreys95_high = beta.ppf(0.975, k_h + 0.5, n_h - k_h + 0.5)

        # Bonferroni-adjusted Wilson limits used to form a >=95% simultaneous interval.
        wilson_adj_low, wilson_adj_high = wilson_interval(k_h, n_h, z_wilson)
    combined_wilson_low += W_h * wilson_adj_low
    combined_wilson_high += W_h * wilson_adj_high

    stratum_rows.append({
        "stratum": stratum,
        "population_N": N_h,
        "sample_n": n_h,
        "confirmed_drone_k": k_h,
        "uncertain_n": unsure_h,
        "precision_confirmed": p_h,
        "weight_W": W_h,
        "wilson95_low": wilson95_low,
        "wilson95_high": wilson95_high,
        "jeffreys95_low": jeffreys95_low,
        "jeffreys95_high": jeffreys95_high,
    })

stratum_results = pd.DataFrame(stratum_rows).set_index("stratum")
cochran_se = np.sqrt(cochran_variance)
cochran_ci_low = max(0.0, weighted_estimate - z_cochran * cochran_se)
cochran_ci_high = min(1.0, weighted_estimate + z_cochran * cochran_se)

# Stratified Jeffreys posterior interval: independently draw each stratum's
# beta posterior, then combine using the known population weights W_h.
N_SIM = 200_000
rng = np.random.default_rng(RANDOM_SEED)
posterior_precision = np.zeros(N_SIM, dtype=float)
for row in stratum_rows:
    k_h = row["confirmed_drone_k"]
    n_h = row["sample_n"]
    W_h = row["weight_W"]
    N_h = populations[row["stratum"]]
    if n_h == N_h:
        posterior_precision += W_h * (k_h / n_h)
    else:
        posterior_precision += W_h * rng.beta(
            k_h + 0.5,
            n_h - k_h + 0.5,
            size=N_SIM,
        )
jeffreys_overall_low, jeffreys_overall_high = np.quantile(
    posterior_precision, [0.025, 0.975]
)

summary = pd.Series({
    "stratified_precision_estimate": weighted_estimate,
    "cochran_standard_error": cochran_se,
    "cochran_95_ci_low": cochran_ci_low,
    "cochran_95_ci_high": cochran_ci_high,
    "stratified_wilson_95_low_bonferroni": combined_wilson_low,
    "stratified_wilson_95_high_bonferroni": combined_wilson_high,
    "stratified_jeffreys_95_low": jeffreys_overall_low,
    "stratified_jeffreys_95_high": jeffreys_overall_high,
    "weighted_uncertain_share": weighted_uncertainty,
    "cohen_kappa_3class_double_annotated": kappa_3class,
    "n_double_annotated_families": len(pairs),
}, name="value")

print("Stratum-level validation:")
display(stratum_results)
print("Overall query validation:")
display(summary.to_frame())

stratum_results.to_csv(VALIDATION_DIR / "validation_precision_by_stratum.csv")
summary.to_csv(VALIDATION_DIR / "validation_precision_overall.csv")
validated_sample.to_csv(VALIDATION_DIR / "validation_sample_with_labels.csv", index=False)

Stratum-level validation:


,population_N,sample_n,confirmed_drone_k,uncertain_n,precision_confirmed,weight_W,wilson95_low,wilson95_high,jeffreys95_low,jeffreys95_high
stratum,,,,,,,,,,
Core Classification,61967,72,37,14,0.513889,0.86992,0.400698,0.625673,0.399804,0.626882
Boundary Rescue,9266,28,26,0,0.928571,0.13008,0.773546,0.980188,0.790053,0.984859


Overall query validation:


,value
stratified_precision_estimate,0.567831
cochran_standard_error,0.051971
cochran_95_ci_low,0.465970
cochran_95_ci_high,0.669691
stratified_wilson_95_low_bonferroni,0.431939
stratified_wilson_95_high_bonferroni,0.685185
stratified_jeffreys_95_low,0.465742
stratified_jeffreys_95_high,0.665369
weighted_uncertain_share,0.169151
cohen_kappa_3class_double_annotated,0.801980
